# 05 — Area GNN Training & Hierarchical Coordination
Trains GNN forecaster on ward data, runs full hierarchical simulation.
**Run 01, 02, 04 first.**

In [ ]:
import sys, os
from pathlib import Path
PROJECT_ROOT = Path(os.getcwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

## 1. Train Area GNN on collected ward data

In [ ]:
import torch
from src.topology import Topology
from src.controllers.area_controller import AreaForecaster

topology = Topology(PROJECT_ROOT)
AREA_ID = 'Basavanagudi'

forecaster = AreaForecaster(AREA_ID, topology)

# Combine GNN data from all wards
gnn_dir = PROJECT_ROOT / 'models' / 'gnn'
all_data = []
for f in gnn_dir.glob('ward_*_gnn_data.pt'):
    data = torch.load(f, weights_only=False)
    all_data.extend(data)
    print(f'  Loaded {len(data)} samples from {f.name}')

if all_data:
    combined_path = gnn_dir / 'combined_training_data.pt'
    torch.save(all_data, combined_path)
    print(f'\nTotal samples: {len(all_data)}')
    losses = forecaster.train_offline(combined_path, epochs=100, save_dir=gnn_dir)
    print(f'Final loss: {losses[-1]:.6f}')
else:
    print('No GNN data found. Train ward agents first (notebook 04).')

## 2. Run Full Hierarchical Simulation

In [ ]:
from src.runtime import run_simulation

result = run_simulation(
    scope='ward', identifier='ward_001',
    project_root=PROJECT_ROOT,
    gui=False, scenario_id='normal', max_ticks=600,
)

import json
print(json.dumps(result, indent=2, default=str))

## 3. Results

In [ ]:
results_dir = PROJECT_ROOT / 'results' / 'inference'
for f in sorted(results_dir.glob('*.json')):
    print(f'  {f.name}')